In [1]:
# Import required libraries
import pandas as pd
from datetime import datetime
import os

print("Libraries loaded successfully")

Libraries loaded successfully


Cell 2 — Create Sample Customer Dataset (with PII)

In [2]:
# Create sample telecom customer data with Personally Identifiable Information (PII)

data = [
    {"customer_id":1001, "name":"Asha Mehta",  "phone":"+91-9876543210",
     "address":"23 Lodi Rd, Delhi",   "plan_type":"Prepaid",  "arpu_inr":180},

    {"customer_id":1002, "name":"Ravi Kumar",  "phone":"+91-9988776655",
     "address":"12 Marine Dr, Mumbai","plan_type":"Postpaid", "arpu_inr":280},

    {"customer_id":1003, "name":"Sneha Rao",   "phone":"+91-9090909090",
     "address":"7 Anna Salai, Chennai","plan_type":"Prepaid", "arpu_inr":210},

    {"customer_id":1004, "name":"Manoj Singh", "phone":"+91-9123456789",
     "address":"44 CR Park, Delhi",   "plan_type":"Postpaid", "arpu_inr":320},

    {"customer_id":1005, "name":"Divya Jain",  "phone":"+91-9000011111",
     "address":"18 Park St, Kolkata", "plan_type":"Prepaid",  "arpu_inr":120}
]

# Convert list to DataFrame
df = pd.DataFrame(data)

# Save dataset to CSV
df.to_csv("customer_data.csv", index=False)

print("customer_data.csv created successfully")

customer_data.csv created successfully


Cell 3 — Define Audit Log Function

In [3]:
# File where access logs will be stored
AUDIT_LOG = "access_audit.log"

# Function to record who accessed the data and which fields were requested
def log_access(role: str, fields_returned: list):
    
    # Create timestamp
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Append access info to log file
    with open(AUDIT_LOG, "a", encoding="utf-8") as f:
        f.write(f"[{ts}] role={role} fields={fields_returned}\n")

Cell 4 — Define Role-Based Masking Policy

In [4]:
# Define what each role can see
# show  → full data visible
# hide  → completely hidden
# mask  → partially visible

POLICY = {

    "junior_analyst": {
        "name": "hide",
        "phone": "hide",
        "address": "hide",
        "arpu_inr": "show",
        "plan_type": "show"
    },

    "senior_analyst": {
        "name": "hide",
        "phone": "mask",
        "address": "hide",
        "arpu_inr": "show",
        "plan_type": "show"
    },

    "manager": {
        "name": "show",
        "phone": "show",
        "address": "show",
        "arpu_inr": "show",
        "plan_type": "show"
    }
}

Cell 5 — Function to Apply Masking

In [5]:
# Function that masks data based on the role policy

def apply_masking(df: pd.DataFrame, role: str) -> pd.DataFrame:
    
    # Check if role exists
    if role not in POLICY:
        raise ValueError(f"Unknown role: {role}")
    
    # Get role policy
    pol = POLICY[role]
    
    # Create copy of dataframe to avoid modifying original data
    out = df.copy()

    # ---- NAME MASKING ----
    if pol.get("name") == "hide" and "name" in out:
        out["name"] = "ANONYMIZED"

    # ---- PHONE MASKING ----
    if "phone" in out:
        if pol.get("phone") == "hide":
            out["phone"] = "XXXXX"
        
        elif pol.get("phone") == "mask":
            # Keep first 3 characters, mask rest
            out["phone"] = out["phone"].astype(str).str[:3] + "-XXX-XXXX"

    # ---- ADDRESS MASKING ----
    if pol.get("address") == "hide" and "address" in out:
        out["address"] = "HIDDEN"

    return out

Cell 6 — Function to Retrieve Role-Based Data

In [6]:
# Main function that returns masked dataset according to role

def get_customer_data(role: str, fields: list | None = None):

    # Read customer dataset
    df = pd.read_csv("customer_data.csv")

    # Always include customer_id
    mandatory = ["customer_id"]

    # If user doesn't specify fields, return all columns
    if fields is None:
        fields = list(df.columns)

    # Ensure customer_id is included and remove duplicates
    fields = list(dict.fromkeys(mandatory + fields))

    # Select requested columns
    df = df[fields]

    # Apply masking based on role
    masked = apply_masking(df, role)

    # Record access event
    log_access(role, fields)

    return masked

Cell 7 — Generate Views for Each Role

In [7]:
# Columns we want to retrieve
cols = ["customer_id", "name", "phone", "address", "plan_type", "arpu_inr"]

print("Junior Analyst View")
jun = get_customer_data("junior_analyst", cols)
display(jun)

# Save result
jun.to_csv("view_junior.csv", index=False)

Junior Analyst View


,customer_id,name,phone,address,plan_type,arpu_inr
0,1001,ANONYMIZED,XXXXX,HIDDEN,Prepaid,180
1,1002,ANONYMIZED,XXXXX,HIDDEN,Postpaid,280
2,1003,ANONYMIZED,XXXXX,HIDDEN,Prepaid,210
3,1004,ANONYMIZED,XXXXX,HIDDEN,Postpaid,320
4,1005,ANONYMIZED,XXXXX,HIDDEN,Prepaid,120


Cell 8 — Senior Analyst View

In [8]:
print("Senior Analyst View")

sen = get_customer_data("senior_analyst", cols)
display(sen)

# Save result
sen.to_csv("view_senior.csv", index=False)

Senior Analyst View


,customer_id,name,phone,address,plan_type,arpu_inr
0,1001,ANONYMIZED,+91-XXX-XXXX,HIDDEN,Prepaid,180
1,1002,ANONYMIZED,+91-XXX-XXXX,HIDDEN,Postpaid,280
2,1003,ANONYMIZED,+91-XXX-XXXX,HIDDEN,Prepaid,210
3,1004,ANONYMIZED,+91-XXX-XXXX,HIDDEN,Postpaid,320
4,1005,ANONYMIZED,+91-XXX-XXXX,HIDDEN,Prepaid,120


Cell 9 — Manager View

In [9]:
print("Manager View")

mgr = get_customer_data("manager", cols)
display(mgr)

# Save result
mgr.to_csv("view_manager.csv", index=False)

Manager View


,customer_id,name,phone,address,plan_type,arpu_inr
0,1001,Asha Mehta,+91-9876543210,"23 Lodi Rd, Delhi",Prepaid,180
1,1002,Ravi Kumar,+91-9988776655,"12 Marine Dr, Mumbai",Postpaid,280
2,1003,Sneha Rao,+91-9090909090,"7 Anna Salai, Chennai",Prepaid,210
3,1004,Manoj Singh,+91-9123456789,"44 CR Park, Delhi",Postpaid,320
4,1005,Divya Jain,+91-9000011111,"18 Park St, Kolkata",Prepaid,120


Cell 10 — Check Audit Log

In [10]:
# Display last few entries of audit log

print("Audit Log Preview")

with open(AUDIT_LOG, "r", encoding="utf-8") as f:
    lines = f.read().splitlines()

print(lines[-3:])

Audit Log Preview
["[2026-03-12 11:10:44] role=junior_analyst fields=['customer_id', 'name', 'phone', 'address', 'plan_type', 'arpu_inr']", "[2026-03-12 11:10:55] role=senior_analyst fields=['customer_id', 'name', 'phone', 'address', 'plan_type', 'arpu_inr']", "[2026-03-12 11:11:05] role=manager fields=['customer_id', 'name', 'phone', 'address', 'plan_type', 'arpu_inr']"]
